<a href="https://colab.research.google.com/github/sahitani/Gen-AI-Project/blob/main/titanic_competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Cell 1 — Setup and load data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Load the uploaded datasets
train_df = pd.read_csv('/Titanic_train.csv')
test_df = pd.read_csv('/Titanic_test.csv')

print("Train data shape:", train_df.shape)
print("Test data shape: ", test_df.shape)

print("\nFirst 3 rows of training data:")
display(train_df.head(3))

print("\nColumns in training data:")
print(train_df.columns.tolist())

Train data shape: (891, 12)
Test data shape:  (418, 11)

First 3 rows of training data:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S



Columns in training data:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [14]:
# Cell 2 — Basic inspection of training data

print("=" * 60)
print("MISSING VALUES IN TRAIN DATA")
print("=" * 60)
missing_train = train_df.isnull().sum()
missing_pct_train = (missing_train / len(train_df) * 100).round(2)
missing_summary_train = pd.DataFrame({
    'Missing Count': missing_train,
    'Missing %': missing_pct_train
})
print(missing_summary_train[missing_summary_train['Missing Count'] > 0])

print("\n" + "=" * 60)
print("MISSING VALUES IN TEST DATA")
print("=" * 60)
missing_test = test_df.isnull().sum()
missing_pct_test = (missing_test / len(test_df) * 100).round(2)
missing_summary_test = pd.DataFrame({
    'Missing Count': missing_test,
    'Missing %': missing_pct_test
})
print(missing_summary_test[missing_summary_test['Missing Count'] > 0])

print("\n" + "=" * 60)
print("TARGET DISTRIBUTION (Survived)")
print("=" * 60)
print(train_df['Survived'].value_counts())
print(f"\nSurvival rate: {train_df['Survived'].mean()*100:.1f}%")

print("\n" + "=" * 60)
print("NUMERIC FEATURES SUMMARY")
print("=" * 60)
print(train_df.describe().round(2))

MISSING VALUES IN TRAIN DATA
          Missing Count  Missing %
Age                 177      19.87
Cabin               687      77.10
Embarked              2       0.22

MISSING VALUES IN TEST DATA
       Missing Count  Missing %
Age               86      20.57
Fare               1       0.24
Cabin            327      78.23

TARGET DISTRIBUTION (Survived)
Survived
0    549
1    342
Name: count, dtype: int64

Survival rate: 38.4%

NUMERIC FEATURES SUMMARY
       PassengerId  Survived  Pclass     Age   SibSp   Parch    Fare
count       891.00    891.00  891.00  714.00  891.00  891.00  891.00
mean        446.00      0.38    2.31   29.70    0.52    0.38   32.20
std         257.35      0.49    0.84   14.53    1.10    0.81   49.69
min           1.00      0.00    1.00    0.42    0.00    0.00    0.00
25%         223.50      0.00    2.00   20.12    0.00    0.00    7.91
50%         446.00      0.00    3.00   28.00    0.00    0.00   14.45
75%         668.50      1.00    3.00   38.00    1.00    0.

In [15]:
# Cell 3 — Survival rates broken down by categorical features

print("=" * 60)
print("SURVIVAL RATE BY SEX")
print("=" * 60)
sex_survival = train_df.groupby('Sex')['Survived'].agg(['mean', 'count']).round(3)
sex_survival.columns = ['Survival Rate', 'Total Passengers']
print(sex_survival)

print("\n" + "=" * 60)
print("SURVIVAL RATE BY PASSENGER CLASS")
print("=" * 60)
class_survival = train_df.groupby('Pclass')['Survived'].agg(['mean', 'count']).round(3)
class_survival.columns = ['Survival Rate', 'Total Passengers']
print(class_survival)

print("\n" + "=" * 60)
print("SURVIVAL RATE BY EMBARKED")
print("=" * 60)
embarked_survival = train_df.groupby('Embarked')['Survived'].agg(['mean', 'count']).round(3)
embarked_survival.columns = ['Survival Rate', 'Total Passengers']
print(embarked_survival)

print("\n" + "=" * 60)
print("SURVIVAL RATE BY SEX AND CLASS COMBINED")
print("=" * 60)
combined_survival = train_df.groupby(['Sex', 'Pclass'])['Survived'].agg(['mean', 'count']).round(3)
combined_survival.columns = ['Survival Rate', 'Total Passengers']
print(combined_survival)

print("\n" + "=" * 60)
print("SURVIVAL RATE BY FAMILY SIZE")
print("=" * 60)
train_df['family_size_temp'] = train_df['SibSp'] + train_df['Parch'] + 1
family_survival = train_df.groupby('family_size_temp')['Survived'].agg(['mean', 'count']).round(3)
family_survival.columns = ['Survival Rate', 'Total Passengers']
print(family_survival)
train_df = train_df.drop('family_size_temp', axis=1)  # cleanup

SURVIVAL RATE BY SEX
        Survival Rate  Total Passengers
Sex                                    
female          0.742               314
male            0.189               577

SURVIVAL RATE BY PASSENGER CLASS
        Survival Rate  Total Passengers
Pclass                                 
1               0.630               216
2               0.473               184
3               0.242               491

SURVIVAL RATE BY EMBARKED
          Survival Rate  Total Passengers
Embarked                                 
C                 0.554               168
Q                 0.390                77
S                 0.337               644

SURVIVAL RATE BY SEX AND CLASS COMBINED
               Survival Rate  Total Passengers
Sex    Pclass                                 
female 1               0.968                94
       2               0.921                76
       3               0.500               144
male   1               0.369               122
       2               0.